# RISE neural network workflow

Training, group testing, and embedding extraction are independent scripts; run only the cells needed for the current step.

## Check PyTorch and GPU

In [ ]:
from pathlib import Path
import socket
import subprocess
import sys
import torch

PROJECT_ROOT = Path('..').resolve()
from rise.neural_network.config import parse_training_config

print('Host:', socket.gethostname())
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f'GPU {index}:', torch.cuda.get_device_name(index))
else:
    print('No GPU is visible; neural-network steps will run on CPU.')

## Scripts and runtime settings

In [ ]:
TRAIN_MODULE = 'rise.neural_network.train'
TEST_MODULE = 'rise.neural_network.test'
TRANSFORM_MODULE = 'rise.neural_network.transform'
CONFIG_FILE = PROJECT_ROOT / 'configs/hcp_network_pretrain_parameters.jsonc'

# All paths, model settings, and optimization parameters come from CONFIG_FILE.
nn_config = parse_training_config(['--config', str(CONFIG_FILE)])
configured_device = nn_config.device
if configured_device.lower() == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
elif configured_device.startswith('cuda') and not torch.cuda.is_available():
    print(
        f'Configured device {configured_device!r} is unavailable; '
        'falling back to CPU.'
    )
    DEVICE = 'cpu'
else:
    DEVICE = configured_device

TRAIN_DATA = nn_config.train_data
TEST_DATA = nn_config.test_data
TEST_GROUP_DATA = nn_config.test_group_data

print('Configuration:', CONFIG_FILE)
print('Run name:', nn_config.run_name)
print('Model name:', nn_config.model_name)
print('Configured device:', configured_device)
print('Runtime device:', DEVICE)

for module in (TRAIN_MODULE, TEST_MODULE, TRANSFORM_MODULE):
    print()
    print(f'### {module}')
    subprocess.run([sys.executable, '-m', module, '--config', str(CONFIG_FILE), '--help'], check=True)

## Train

This step trains neural network models and saves in every `repeat_*` model directory.

In [ ]:
if not TRAIN_DATA.is_file():
    raise FileNotFoundError(f'Missing training input: {TRAIN_DATA}')

subprocess.run([
    sys.executable, '-m', TRAIN_MODULE,
    '--config', str(CONFIG_FILE),
    '--device', DEVICE,
], check=True)


## Model evaluation

Validates parcel classification accuracy on the independent test set and saves predicted region labels, along with per-atlas accuracy for every repeat.

In [ ]:
# By default, each repeat loads best_validloss_epoch.pth.
if not TEST_GROUP_DATA.is_file():
    raise FileNotFoundError(f'Missing group test input: {TEST_GROUP_DATA}')

subprocess.run([
    sys.executable, '-m', TEST_MODULE,
    '--config', str(CONFIG_FILE),
    '--device', DEVICE,
], check=True)


## Extract parcel embeddings

Set `TRANSFORM_SPLIT` to `train`, `test`, or `both`. This step uses the trained models to extract latent space embeddings for all individual subjects' parcels in the training set, test set, or both.
Each repeat writes left/right HDF5 files beneath its `results` directory.

In [ ]:
TRANSFORM_SPLIT = 'both'
for data_path in (TRAIN_DATA, TEST_DATA):
    if not data_path.is_file():
        raise FileNotFoundError(f'Missing transform input: {data_path}')

subprocess.run([
    sys.executable, '-m', TRANSFORM_MODULE,
    '--config', str(CONFIG_FILE),
    '--device', DEVICE,
    '--split', TRANSFORM_SPLIT,
], check=True)
